In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import torch
import torch.nn as nn
from source.data import trainLoader
from torch.utils.data import DataLoader
from source.model import ResNextModel
from source.score import scoreModel
from apex import amp

In [3]:
def score(fold):
    loader = {}
    loader['image_path'] = '../../data/train/'
    loader['label_path'] = '../../data/train.csv'
    loader['fold_idx'] = fold
    _, score = trainLoader(**loader)
    score = DataLoader(score, batch_size=10, shuffle=False, num_workers=6, drop_last=False)
    model = ResNextModel()
    weights = torch.load('../../model/model_{}.pt'.format(fold), map_location='cpu')
    model.load_state_dict(weights['model_state_dict'])
    model = model.to('cuda:0')
    model = amp.initialize(model, opt_level="O2",keep_batchnorm_fp32=True, verbosity=0)
    data = scoreModel(model, score)
    data.to_csv('../../valid/valid_{}.csv'.format(fold), index=False)
    model = model.cpu()
    del model, data
    return None

In [4]:
score(1)

Train Images: 539406 Valid Images: 134852


In [5]:
score(2)

Train Images: 539406 Valid Images: 134852


In [6]:
score(3)

In [7]:
score(4)

Train Images: 539407 Valid Images: 134851


In [8]:
score(5)

Train Images: 539407 Valid Images: 134851
